# 🎯 Logistic Regression

Despite having the word "Regression" in its name, Logistic Regression is actually a **Classification** algorithm!

While Linear Regression predicts a continuous number (like salary), Logistic Regression predicts a category (like "Spam" or "Not Spam", "Dog" or "Cat", "Will Buy" or "Won't Buy").

## 🧠 1. The Theory: Why not just use Linear Regression?

Imagine trying to predict if someone will buy a luxury car based on their wealth. The answer is either **0 (No)** or **1 (Yes)**.

If we draw a straight line (Linear Regression) through the data:
1. **Outliers ruin it:** A single billionaire in the dataset will pull the line so far to the right that normal rich people will be predicted as '0' instead of '1'.
2. **Nonsense predictions:** A straight line goes on forever. For a poor person, it might predict a `-0.5` chance of buying. For a billionaire, it might predict a `3.5` chance. Probabilities must be between 0 and 1!

**The Solution:** We need to bend that straight line into an "S" shape that flatlines at 0 and flatlines at 1. This is the **Sigmoid Curve**.

## 🧮 2. The Math (The Sigmoid Function)

We take the exact same equation from Linear Regression ($y = mx + b$), but instead of using the raw output, we wrap it inside a special function called the **Sigmoid Function**:

$$ \sigma(z) = \frac{1}{1 + e^{-z}} $$

Where $z = \beta_0 + \beta_1x_1 + \dots + \beta_nx_n$.

**What does this do?**
No matter how big or small $z$ gets, the formula forces the output to be a number between **0 and 1**.
- If output is $\ge 0.5$, we classify it as **Class 1 (Yes)**.
- If output is $< 0.5$, we classify it as **Class 0 (No)**.

## 💻 3. The Implementation

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import confusion_matrix, accuracy_score, classification_report

plt.style.use('seaborn-v0_8-whitegrid')

In [ ]:
# 1. Generate Synthetic Data: Predicting if a user bought an SUV based on Age and Salary
np.random.seed(42)
n_samples = 300

age = np.random.normal(40, 10, n_samples)
salary = np.random.normal(60000, 20000, n_samples)

# The probability of buying increases heavily with age and salary
z = -10 + (0.15 * age) + (0.00005 * salary)
probability = 1 / (1 + np.exp(-z)) # The Sigmoid function in action!

purchased = (probability > np.random.rand(n_samples)).astype(int)

df = pd.DataFrame({'Age': age, 'Salary': salary, 'Purchased': purchased})
print(df.head())

In [ ]:
# 2. Visualize the Data
plt.figure(figsize=(8, 6))
sns.scatterplot(x='Age', y='Salary', hue='Purchased', data=df, palette='bwr', alpha=0.8)
plt.title('Customer Data: SUV Purchase')
plt.show()
print('Notice how older, wealthier people (top right) are more likely to buy (red).')

In [ ]:
# 3. Prepare the Data
X = df[['Age', 'Salary']]
y = df['Purchased']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, random_state=42)

# VERY IMPORTANT: Distance-based algorithms and algorithms that use Gradient Descent
# (like Logistic Regression) need Feature Scaling! Age is ~40, Salary is ~60000.
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

In [ ]:
# 4. Train the Logistic Regression Model
classifier = LogisticRegression(random_state=42)
classifier.fit(X_train_scaled, y_train)

# 5. Make Predictions
y_pred = classifier.predict(X_test_scaled)

## 🧐 4. Evaluation: The Confusion Matrix

In Regression, we used R-Squared. In Classification, our best friend is the **Confusion Matrix**.
It tells us exactly *how* our model got confused.

In [ ]:
# 6. Generate the Confusion Matrix
cm = confusion_matrix(y_test, y_pred)

plt.figure(figsize=(6, 4))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', cbar=False)
plt.xlabel('Predicted Label')
plt.ylabel('True Label')
plt.title('Confusion Matrix')
plt.show()

**How to read this matrix:**
- **Top-Left (True Negatives)**: People who didn't buy, and we correctly predicted they wouldn't.
- **Bottom-Right (True Positives)**: People who did buy, and we correctly predicted they would.
- **Top-Right (False Positives - Type 1 Error)**: We predicted they would buy, but they actually didn't.
- **Bottom-Left (False Negatives - Type 2 Error)**: We predicted they wouldn't buy, but they actually did!

In [ ]:
# 7. Classification Report
print("Accuracy:", accuracy_score(y_test, y_pred))
print("\nDetailed Report:\n", classification_report(y_test, y_pred))

### The Metrics Dictionary
- **Accuracy**: Overall, how often was the model correct? `(TP + TN) / Total`
- **Precision**: When it predicts someone will buy, how often is it right? `TP / (TP + FP)`. (Important when false positives are costly, e.g., spam filters)
- **Recall**: Out of all the people who actually bought, how many did we find? `TP / (TP + FN)`. (Important when false negatives are deadly, e.g., cancer detection)

## 🚧 Visualizing the Decision Boundary

Logistic Regression essentially draws a straight line through the data. Everything on one side is classified as 0, and everything on the other is 1.

In [ ]:
# Create a meshgrid to plot the decision boundary
X_set, y_set = X_test_scaled, y_test.values
X1, X2 = np.meshgrid(np.arange(start = X_set[:, 0].min() - 1, stop = X_set[:, 0].max() + 1, step = 0.01),
                     np.arange(start = X_set[:, 1].min() - 1, stop = X_set[:, 1].max() + 1, step = 0.01))
plt.figure(figsize=(8, 6))
plt.contourf(X1, X2, classifier.predict(np.array([X1.ravel(), X2.ravel()]).T).reshape(X1.shape),
             alpha = 0.5, cmap = 'bwr')
plt.scatter(X_set[y_set == 0, 0], X_set[y_set == 0, 1], color = 'blue', label = '0 (Did not buy)')
plt.scatter(X_set[y_set == 1, 0], X_set[y_set == 1, 1], color = 'red', label = '1 (Bought)')
plt.title('Logistic Regression Decision Boundary')
plt.xlabel('Age (Scaled)')
plt.ylabel('Salary (Scaled)')
plt.legend()
plt.show()
print('The line separating the red zone and blue zone is the Decision Boundary!')

## 🎉 Congratulations!
You've mastered Logistic Regression, the foundation of Classification! You also learned about Sigmoid functions, Feature Scaling, and the Confusion Matrix.
Next up: we'll look at a completely different way to classify data using **K-Nearest Neighbors (KNN)**!